# Dice Calculation From Binary Spot Labels

`inputs/mask_hypo.csv` contains the independent manual annotations used as the reference for Dice comparison.
Each cutoff is evaluated against this same fixed mask.
Input provenance is documented in README.md and provenance.json.

# 1. Set Up

Run from this notebook's directory. The six input CSV files contain only
sample ID, spot ID, and `is_hypo` (0 or 1).

In [ ]:
from pathlib import Path
import platform
import numpy as np
import pandas as pd
from IPython.display import display
from calculate_dice import read_labels, align_labels, save_results

INPUT_DIR = Path('inputs')
OUTPUT_DIR = Path('results_notebook')
CUTOFFS = [30, 35, 40, 45, 50]
EXAMPLE_SAMPLE = 'UKF248_T_1'
EXAMPLE_CUTOFF = 40
pd.options.display.float_format = '{:.6f}'.format
print({'Python': platform.python_version(), 'NumPy': np.__version__, 'pandas': pd.__version__})

# 2. Read the Reference Mask

The same reference mask is used for every cutoff. Both positive and negative
spots are retained.

In [ ]:
reference = read_labels(INPUT_DIR / 'mask_hypo.csv')
display(reference.head(8))
sample_sizes = reference.groupby('sample').agg(
    n_spots=('spot_id', 'size'), n_mask_hypo=('is_hypo', 'sum'))
display(sample_sizes)
print('Samples:', reference['sample'].nunique(), '| Total spots:', len(reference))

# 3. Read the Five Hypo Label Tables

These are supplied processed PAN region memberships (`PAN_core`), not raw
abundance thresholds. This calculation reads the existing binary labels;
it does not recalculate thresholds or alter regions.

In [ ]:
predictions = {cutoff: read_labels(INPUT_DIR / f'hypo_{cutoff}.csv') for cutoff in CUTOFFS}
display(predictions[EXAMPLE_CUTOFF].head(8))
display(pd.DataFrame([
    {'cutoff_percent': cutoff, 'n_spots': len(frame), 'n_prediction_hypo': int(frame.is_hypo.sum())}
    for cutoff, frame in predictions.items()
]))

# 4. Align and Validate Spots

Align on `(sample, spot_id)`, not row position. Missing, extra, duplicate,
or non-binary rows cause an error. No spots are removed by an inner join.

In [ ]:
aligned = {cutoff: align_labels(reference, frame) for cutoff, frame in predictions.items()}
alignment_checks = pd.DataFrame([
    {'cutoff_percent': cutoff, 'n_spots': len(frame), 'n_samples': frame['sample'].nunique(),
     'all_reference_spots_retained': len(frame) == len(reference)}
    for cutoff, frame in aligned.items()
])
display(alignment_checks)
display(aligned[EXAMPLE_CUTOFF].head(8))

# 5. Calculate Dice for One Sample

Let $A$ be mask-positive spots and $B$ be predicted Hypo-positive spots.

$$\mathrm{Dice}=\frac{2|A\cap B|}{|A|+|B|}
=\frac{2TP}{2TP+FP+FN}.$$

If both positive sets are empty, report NaN. True negatives do not enter Dice.

In [ ]:
example = aligned[EXAMPLE_CUTOFF].query('sample == @EXAMPLE_SAMPLE')
a = example['mask_is_hypo'].to_numpy(dtype=bool)
b = example['prediction_is_hypo'].to_numpy(dtype=bool)
tp = int((a & b).sum())
fp = int((~a & b).sum())
fn = int((a & ~b).sum())
tn = int((~a & ~b).sum())
denominator = int(a.sum() + b.sum())
dice = 2 * tp / denominator if denominator else np.nan
display(pd.DataFrame([{'sample': EXAMPLE_SAMPLE, 'cutoff_percent': EXAMPLE_CUTOFF,
    'n_spots': len(a), 'n_mask_hypo': int(a.sum()), 'n_prediction_hypo': int(b.sum()),
    'intersection': tp, 'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
    'denominator': denominator, 'dice': dice}]))
print(f'Dice = 2 * {tp} / ({a.sum()} + {b.sum()}) = {dice:.9f}')

# 6. Calculate All Samples and Cutoffs

Repeat the same formula independently for each sample and cutoff.

In [ ]:
records = []
for cutoff, frame in aligned.items():
    for sample, group in frame.groupby('sample', sort=True):
        a = group['mask_is_hypo'].to_numpy(dtype=bool)
        b = group['prediction_is_hypo'].to_numpy(dtype=bool)
        tp, fp, fn, tn = (int(values.sum()) for values in [a & b, ~a & b, a & ~b, ~a & ~b])
        denominator = int(a.sum() + b.sum())
        records.append({'sample': sample, 'cutoff_percent': cutoff, 'n_spots': len(a),
            'n_mask_hypo': int(a.sum()), 'n_prediction_hypo': int(b.sum()),
            'intersection': tp, 'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
            'denominator': denominator, 'dice': 2 * tp / denominator if denominator else np.nan,
            'dice_defined': denominator > 0})
per_sample = pd.DataFrame(records)
display(per_sample)
assert len(per_sample) == reference['sample'].nunique() * len(CUTOFFS)
assert per_sample.dice_defined.all(), 'An empty-positive comparison needs explicit handling.'

# 7. Summarize Across Samples

Give each sample equal weight: mean Dice = sum of sample Dice / number of
defined samples. This is not a pooled-spot Dice. Individual sample peaks
need not coincide with the peak of the mean.

In [ ]:
summary = per_sample.groupby('cutoff_percent', as_index=False).agg(
    n_samples=('sample', 'size'), n_defined=('dice', 'count'), mean_dice=('dice', 'mean'),
    sd_dice=('dice', 'std'), min_dice=('dice', 'min'), max_dice=('dice', 'max'))
wide = per_sample.pivot(index='sample', columns='cutoff_percent', values='dice')
wide.loc['Equal_sample_mean'] = wide.mean(axis=0)
display(summary)
display(wide)
ranked = summary.sort_values('mean_dice', ascending=False)
best, runner_up = ranked.iloc[0], ranked.iloc[1]
print(f'Highest mean: cutoff {int(best.cutoff_percent)}%, Dice = {best.mean_dice:.9f}')
print(f'Gap to next-highest mean = {best.mean_dice - runner_up.mean_dice:.9f}')

# 8. Export Results

Save per-sample counts, the equally weighted summary, and the wide Dice table.

In [ ]:
saved_summary, saved_wide = save_results(per_sample, OUTPUT_DIR)
pd.testing.assert_frame_equal(summary, saved_summary)
pd.testing.assert_frame_equal(wide, saved_wide)
alignment_checks.to_csv(OUTPUT_DIR / 'alignment_checks.csv', index=False)
for path in sorted(OUTPUT_DIR.glob('*.csv')):
    print(path)